# Google Colab: Temporal Fusion Transformer (TFT) Demand Forecasting Workflow
**Project:** Nationwide Demand Forecasting and Inventory Optimization
**Dataset:** M5 Retail Forecasting
**Purpose:** Deep learning sequence-to-sequence forecasting with interpretable multi-head attention and multi-quantile prediction intervals.

> **Instructions:**
> 1. In Google Colab, select `Runtime` > `Change runtime type` > `T4 GPU`.
> 2. Run cells sequentially from top to bottom.
> 3. Exported artifacts (`artifacts/tft/`) can be copied back to the main repository to enable Mode 1 (checkpoint) and Mode 2 (exported predictions) in the application dashboard.

In [1]:
# CELL 1: ENVIRONMENT INFORMATION
import sys, os, platform, psutil
print('Python Version:', sys.version)
print('OS Platform:', platform.platform())
print('CPU Count:', psutil.cpu_count(logical=True))
print('Available RAM (GB):', round(psutil.virtual_memory().total / (1024**3), 2))

try:
    import torch
    print('PyTorch Version:', torch.__version__)
    print('CUDA Available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU Device:', torch.cuda.get_device_name(0))
        print('Device Memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
    else:
        print('No GPU detected. Training will run in CPU-mode (reduced batch size recommended).')
except ImportError:
    print('PyTorch is not yet installed in this runtime.')

Python Version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
OS Platform: Linux-6.6.122+-x86_64-with-glibc2.39
CPU Count: 2
Available RAM (GB): 12.67
PyTorch Version: 2.11.0+cu130
CUDA Available: True
GPU Device: Tesla T4
Device Memory (GB): 14.56


In [2]:
# CELL 2: DEPENDENCY INSTALLATION
# Installs PyTorch Forecasting and PyTorch Lightning with compatible ecosystem libraries
!pip install -q lightning pytorch-forecasting pandas numpy scikit-learn matplotlib

import pytorch_forecasting
import lightning.pytorch as pl
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss
print('PyTorch Lightning Version:', pl.__version__)
print('PyTorch Forecasting Version:', pytorch_forecasting.__version__)

PyTorch Lightning Version: 2.6.6
PyTorch Forecasting Version: 1.8.0


In [3]:
# CELL 3: DATASET LOCATION & PATH CONFIGURATION
import os
from pathlib import Path

# In Google Colab, you can upload files directly via the folder icon on the left panel (/content/)
# Candidate search locations for M5 data files:
SEARCH_DIRS = [
    Path('/content'),
    Path('/content/data'),
    Path('/content/drive/MyDrive')
]
print('Search paths configured:', [str(p) for p in SEARCH_DIRS])

Search paths configured: ['/content', '/content/data', '/content/drive/MyDrive']


In [4]:
# CELL 4: DATA LOADING & PREPARATION SCRIPT
# Expected file to upload from your project:
#   EITHER: sales_train_validation.csv (from data/raw/)
#   OR:     sample_sales.parquet       (from data/sample/, much faster upload ~500 KB)
import pandas as pd, numpy as np
from pathlib import Path

found_file = None
file_type = None

for directory in SEARCH_DIRS:
    parquet_candidate = directory / 'sample_sales.parquet'
    csv_candidate = directory / 'sales_train_validation.csv'
    if parquet_candidate.exists():
        found_file = parquet_candidate
        file_type = 'parquet'
        break
    elif csv_candidate.exists():
        found_file = csv_candidate
        file_type = 'csv'
        break

if not found_file:
    raise FileNotFoundError(
        'File not found! Please upload one of the following to Google Colab:\n'
        '  1. sales_train_validation.csv (from your project data/raw/ folder), OR\n'
        '  2. sample_sales.parquet (from your project data/sample/ folder - recommended for speed).\n\n'
        'How to upload: Click the Folder icon 📁 on the left toolbar in Colab and drag the file in.'
    )

print(f'Successfully found dataset file: {found_file}')
if file_type == 'parquet':
    df = pd.read_parquet(found_file)
else:
    print('Loading raw sales_train_validation.csv (sampling first 500 series for rapid training)...')
    raw_sales = pd.read_csv(found_file, nrows=500)
    d_cols = [c for c in raw_sales.columns if c.startswith('d_')][-180:] # last 180 days for demo
    id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
    df = pd.melt(raw_sales, id_vars=id_vars, value_vars=d_cols, var_name='d', value_name='sales')
    df['sales'] = df['sales'].astype(float)
    d_nums = df['d'].str.replace('d_', '').astype(int)
    df['date'] = pd.to_datetime('2016-01-01') + pd.to_timedelta(d_nums - d_nums.min(), unit='D')

print(f'Loaded dataset with {len(df)} rows and {df["id"].nunique()} unique series.')
print(df.head())

Successfully found dataset file: /content/sales_train_validation.csv
Loading raw sales_train_validation.csv (sampling first 500 series for rapid training)...
Loaded dataset with 90000 rows and 500 unique series.
                              id        item_id    dept_id   cat_id store_id  \
0  HOBBIES_1_001_CA_1_validation  HOBBIES_1_001  HOBBIES_1  HOBBIES     CA_1   
1  HOBBIES_1_002_CA_1_validation  HOBBIES_1_002  HOBBIES_1  HOBBIES     CA_1   
2  HOBBIES_1_003_CA_1_validation  HOBBIES_1_003  HOBBIES_1  HOBBIES     CA_1   
3  HOBBIES_1_004_CA_1_validation  HOBBIES_1_004  HOBBIES_1  HOBBIES     CA_1   
4  HOBBIES_1_005_CA_1_validation  HOBBIES_1_005  HOBBIES_1  HOBBIES     CA_1   

  state_id       d  sales       date  
0       CA  d_1734    1.0 2016-01-01  
1       CA  d_1734    0.0 2016-01-01  
2       CA  d_1734    1.0 2016-01-01  
3       CA  d_1734    1.0 2016-01-01  
4       CA  d_1734    3.0 2016-01-01  


In [5]:
# CELL 5: TIME-SERIES DATASET SPECIFICATION
df['time_idx'] = (df['date'] - df['date'].min()).dt.days
df['dayofweek'] = df['date'].dt.dayofweek.astype(str)
df['month'] = df['date'].dt.month.astype(str)
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(str)
for col in ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']:
    df[col] = df[col].astype(str)

max_prediction_length = 28
max_encoder_length = 60
training_cutoff = df['time_idx'].max() - max_prediction_length

print('Max Encoder Length:', max_encoder_length, 'days')
print('Prediction Horizon:', max_prediction_length, 'days')
print('Training Cutoff Day:', training_cutoff)

Max Encoder Length: 60 days
Prediction Horizon: 28 days
Training Cutoff Day: 151


In [7]:
# CELL 6: CHRONOLOGICAL SPLITTING & DATASET BUILDING
from pytorch_forecasting.data.encoders import NaNLabelEncoder

training_data = df[df['time_idx'] <= training_cutoff].copy()

# Configure NaNLabelEncoder with add_nan=True so unseen categories in validation do not raise KeyError
cat_cols = ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'dayofweek', 'month', 'is_weekend']
encoders = {c: NaNLabelEncoder(add_nan=True) for c in cat_cols}

training = TimeSeriesDataSet(
    training_data,
    time_idx='time_idx',
    target='sales',
    group_ids=['id'],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    time_varying_known_categoricals=['dayofweek', 'month', 'is_weekend'],
    time_varying_unknown_reals=['sales'],
    categorical_encoders=encoders,
    target_normalizer=pytorch_forecasting.data.encoders.TorchNormalizer(method='standard'),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True
)

validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)
batch_size = 64
train_dataloader = training.to_dataloader(batch_size=batch_size, shuffle=True, num_workers=0)
val_dataloader = validation.to_dataloader(batch_size=batch_size * 2, shuffle=False, num_workers=0)
print(f'Train batches: {len(train_dataloader)}, Val batches: {len(val_dataloader)}')


Train batches: 1398, Val batches: 3


/usr/local/lib/python3.13/dist-packages/pytorch_forecasting/data/encoders.py:402: UserWarning: Found 1 unknown classes which were set to NaN
  warnings.warn(


In [8]:
# CELL 7: TEMPORAL FUSION TRANSFORMER INITIALIZATION & TRAINING
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

pl.seed_everything(42)
checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1, filename='best_tft')
early_stop_callback = EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=5, verbose=True, mode='min')

trainer = pl.Trainer(
    max_epochs=10,
    accelerator='auto',
    gradient_clip_val=0.1,
    limit_train_batches=30,
    callbacks=[early_stop_callback, checkpoint_callback],
    enable_progress_bar=True
)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=2,
    dropout=0.1,
    hidden_continuous_size=8,
    loss=QuantileLoss([0.1, 0.5, 0.9]),
    reduce_on_plateau_patience=4
)
print('Model parameter count:', sum(p.numel() for p in tft.parameters()))
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during c

Model parameter count: 26899


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │  8.1 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │     80 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  2.5 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │  1.5 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │    900 │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  2.2 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │    544 │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     32 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  1.4 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │    808 │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │    576 │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  1.1 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │    576 │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     51 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 26.9 K                                                                                           
Non-trainable params: 0                                                                                            
Total params: 26.9 K                                                                                               
Total estimated model params size (MB): 0.108                                                                      
Modules in train mode: 297                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/loops/fit_loop.py:321: The number of training batches 
(30) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if
you want to see logs for the training epoch.

INFO: Metric val_loss improved. New best score: 0.948
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved. New best score: 0.948
INFO: Metric val_loss improved by 0.023 >= min_delta = 0.0001. New best score: 0.925
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.023 >= min_delta = 0.0001. New best score: 0.925
INFO: Metric val_loss improved by 0.136 >= min_delta = 0.0001. New best score: 0.789
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.136 >= min_delta = 0.0001. New best score: 0.789
INFO: Metric val_loss improved by 0.025 >= min_delta = 0.0001. New best score: 0.764
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.025 >= min_delta = 0.0001. New best score: 0.764
INFO: Metric val_loss improved by 0.042 >= min_delta = 0.0001. New best score: 0.722
INFO:lightning.pytorch.callbacks.early_stopping:Metric val_loss improved by 0.042 >= min_delta = 0.0001. New best score: 0.722
IN

In [9]:
# CELL 8: BASELINE COMPARISON (SEASONAL-NAIVE)
test_cutoff = training_cutoff
test_data = df[df['time_idx'] > test_cutoff].copy()
baseline_preds = []
for series_id, grp in test_data.groupby('id'):
    train_hist = df[(df['id'] == series_id) & (df['time_idx'] <= test_cutoff)].sort_values('time_idx')
    last_7 = train_hist['sales'].values[-7:]
    reps = int(np.ceil(len(grp) / 7))
    pred_cycle = np.tile(last_7, reps)[:len(grp)]
    grp = grp.copy()
    grp['baseline_pred'] = pred_cycle
    baseline_preds.append(grp)
baseline_df = pd.concat(baseline_preds, ignore_index=True)
print('Generated Seasonal-Naive baseline predictions for identical test window.')

Generated Seasonal-Naive baseline predictions for identical test window.


In [10]:
# CELL 9: EVALUATION METRICS (WAPE, RMSE, MAE, BIAS)
raw_predictions = tft.predict(val_dataloader, mode='quantiles', return_x=True)
preds_p50 = raw_predictions.output[..., 1].cpu().numpy().flatten()
preds_p10 = raw_predictions.output[..., 0].cpu().numpy().flatten()
preds_p90 = raw_predictions.output[..., 2].cpu().numpy().flatten()
actuals = raw_predictions.x['decoder_target'].cpu().numpy().flatten()

def compute_metrics(act, pred):
    wape = np.sum(np.abs(act - pred)) / max(1e-5, np.sum(act))
    rmse = np.sqrt(np.mean((act - pred)**2))
    mae = np.mean(np.abs(act - pred))
    bias = np.mean(pred - act)
    return {'wape': round(float(wape), 4), 'rmse': round(float(rmse), 4), 'mae': round(float(mae), 4), 'bias': round(float(bias), 4)}

tft_metrics = compute_metrics(actuals, preds_p50)
base_metrics = compute_metrics(baseline_df['sales'].values, baseline_df['baseline_pred'].values)
print('TFT Metrics:', tft_metrics)
print('Seasonal-Naive Metrics:', base_metrics)

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: 💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:lightning.pytorch

TFT Metrics: {'wape': 0.8886, 'rmse': 2.5024, 'mae': 0.9288, 'bias': -0.4789}
Seasonal-Naive Metrics: {'wape': 1.0873, 'rmse': 2.9741, 'mae': 1.1366, 'bias': -0.1319}


In [11]:
# CELL 10: UNCERTAINTY & INTERVAL COVERAGE EVALUATION
in_interval = (actuals >= preds_p10) & (actuals <= preds_p90)
empirical_coverage = float(np.mean(in_interval))
avg_width = float(np.mean(preds_p90 - preds_p10))
interval_summary = {
    'nominal_coverage': 0.80,
    'empirical_coverage': round(empirical_coverage, 4),
    'average_width': round(avg_width, 4),
    'sample_count': int(len(actuals))
}
print('Prediction Interval Quality (80% Nominal):', interval_summary)

Prediction Interval Quality (80% Nominal): {'nominal_coverage': 0.8, 'empirical_coverage': 0.903, 'average_width': 2.5943, 'sample_count': 14000}


In [12]:
# CELL 11: EXPORT ARTIFACTS FOR MAIN APPLICATION INTEGRATION
import json, shutil
from pathlib import Path

export_dir = Path('/content/artifacts/tft')
export_dir.mkdir(parents=True, exist_ok=True)

# 1. Save checkpoint path
ckpt_path = trainer.checkpoint_callback.best_model_path
if ckpt_path and os.path.exists(ckpt_path):
    shutil.copy(ckpt_path, export_dir / 'model.ckpt')
    print('Copied model checkpoint to:', export_dir / 'model.ckpt')

# 2. Export test predictions DataFrame
export_df = baseline_df[['id', 'date', 'sales']].copy()
export_df['predicted'] = preds_p50[:len(export_df)]
export_df['pred_lower'] = preds_p10[:len(export_df)]
export_df['pred_upper'] = preds_p90[:len(export_df)]
export_df.to_csv(export_dir / 'predictions.csv', index=False)

# 3. Export metrics, config & metadata
with open(export_dir / 'evaluation_metrics.json', 'w') as f:
    json.dump({'tft': tft_metrics, 'baseline': base_metrics, 'intervals': interval_summary}, f, indent=2)

with open(export_dir / 'model_config.json', 'w') as f:
    json.dump({'hidden_size': 16, 'attention_head_size': 2, 'quantiles': [0.1, 0.5, 0.9]}, f, indent=2)

with open(export_dir / 'training_metadata.json', 'w') as f:
    json.dump({
        'model_type': 'TemporalFusionTransformer',
        'runtime': 'Google Colab T4 GPU',
        'status': 'COMPLETED_SUCCESSFULLY',
        'epochs_trained': trainer.current_epoch,
        'best_val_loss': float(trainer.checkpoint_callback.best_model_score or 0.0)
    }, f, indent=2)

print('All artifacts successfully exported to:', export_dir.resolve())

Copied model checkpoint to: /content/artifacts/tft/model.ckpt
All artifacts successfully exported to: /content/artifacts/tft


In [13]:
# CELL 12: REPRODUCIBILITY & ENVIRONMENT DUMP
print('=== REPRODUCIBILITY RECORD ===')
print('PyTorch:', torch.__version__)
print('PyTorch Lightning:', pl.__version__)
print('PyTorch Forecasting:', pytorch_forecasting.__version__)
print('Seed: 42')
print('Target: M5 Sales Volume (Non-negative)')

=== REPRODUCIBILITY RECORD ===
PyTorch: 2.11.0+cu130
PyTorch Lightning: 2.6.6
PyTorch Forecasting: 1.8.0
Seed: 42
Target: M5 Sales Volume (Non-negative)


In [14]:
# CELL 13: FINAL VERIFIED SUMMARY
print('===================================================')
print('TFT COLAB EXPERIMENT COMPLETION SUMMARY')
print('===================================================')
print('1. Training Finished: YES')
print('2. TFT Test WAPE:', tft_metrics['wape'], '(Seasonal-Naive:', base_metrics['wape'], ')')
print('3. Empirical Interval Coverage:', f"{interval_summary['empirical_coverage'] * 100:.1f}%")
print('4. Artifacts Exported: model.ckpt, predictions.csv, evaluation_metrics.json, training_metadata.json')
print('Ready for integration into main dashboard!')

TFT COLAB EXPERIMENT COMPLETION SUMMARY
1. Training Finished: YES
2. TFT Test WAPE: 0.8886 (Seasonal-Naive: 1.0873 )
3. Empirical Interval Coverage: 90.3%
4. Artifacts Exported: model.ckpt, predictions.csv, evaluation_metrics.json, training_metadata.json
Ready for integration into main dashboard!
